# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## Nombe: Jesua Villacis
## Fecha: 18/11/2026


In [47]:
# Dependencias
import re, math, heapq, random
from collections import Counter, defaultdict

import json
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from datasets import load_dataset
from scipy.stats import wilcoxon

In [48]:
# Preparación del corpus
DATASET = "BeIR/nfcorpus"
DATASET_QRELS = "BeIR/nfcorpus-qrels"
SPLIT_QRELS = "test"
K1, B = 1.5, 0.75          # parámetros BM25
K_EVAL = 10

def cargar_corpus():
    ds = load_dataset(DATASET, "corpus")["corpus"]
    return {
        str(f["_id"]): {
            "titulo": (f.get("title") or "").strip(),
            "texto": (f.get("text") or "").strip(),
        }
        for f in ds
    }

def cargar_qrels(split=SPLIT_QRELS):
    ds = load_dataset(DATASET_QRELS)[split]
    qrels = defaultdict(dict)
    for f in ds:
        if int(f["score"]) > 0:
            qrels[str(f["query-id"])][str(f["corpus-id"])] = int(f["score"])
    return dict(qrels)

def cargar_consultas(ids_validos):
    ds = load_dataset(DATASET, "queries")["queries"]
    return {str(f["_id"]): f["text"].strip() for f in ds if str(f["_id"]) in ids_validos}

def texto_documento(doc):
    return f"{doc['titulo']} {doc['texto']}".strip()

corpus = cargar_corpus()
qrels = cargar_qrels()

qrels = {q: {d: r for d, r in docs.items() if d in corpus} for q, docs in qrels.items()}
qrels = {q: docs for q, docs in qrels.items() if docs}
consultas = cargar_consultas(set(qrels))
qrels = {q: d for q, d in qrels.items() if q in consultas}


In [49]:
# Preprocesamiento
nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))
_stemmer = PorterStemmer()
_PATRON = re.compile(r"[a-z0-9]+")
_cache = {}

def _stem(t):
    if t not in _cache:
        _cache[t] = _stemmer.stem(t)
    return _cache[t]

def tokenizar(texto):
    return _PATRON.findall(texto.lower())

def preprocesar(texto):
    return [_stem(t) for t in tokenizar(texto) if t not in STOPWORDS and len(t) > 1]


### Desiciónes sobre el preprocesamiento
- Se normalizó las palabras de los documentos a minusculas ya que estos son en idioma inglés por lo que no se tiene que tener en cuenta acentos como en el español.
- Se tokenizo a los documentos por palabras mediante una expresión regular para así tambien tomar en cuenta números y excluir puntuaciones.
- Se eliminaron stopwords mediante la lista predefinida de nltk para el idioma inglés.
- Se utilizó el stemmer de porter ya que se utilizó tambien en el proyecto y porque es el mas usado para realizar stemming en ingles.


In [50]:
# Indice invertido
class IndiceInvertido:
    def __init__(self):
        self.postings = {}
        self.longitudes = {}       # Tokens por documento
        self.n_docs = 0
        self.longitud_promedio = 0.0

    def construir(self, docs_tokens):
        postings = defaultdict(dict)
        for doc_id, tokens in docs_tokens.items():
            self.longitudes[doc_id] = len(tokens)
            for termino, tf in Counter(tokens).items():
                postings[termino][doc_id] = tf
        self.postings = dict(postings)
        self.n_docs = len(self.longitudes)
        self.longitud_promedio = sum(self.longitudes.values()) / self.n_docs
        return self

    def df(self, termino):
        return len(self.postings.get(termino, ()))

    def documentos_con(self, termino):
        return self.postings.get(termino, {})

docs_tokens = {d: preprocesar(texto_documento(doc)) for d, doc in corpus.items()}
indice = IndiceInvertido().construir(docs_tokens)

print(f"Docs indexados: {indice.n_docs} | Vocabulario: {len(indice.postings)} | "
      f"Longitud promedio: {indice.longitud_promedio:.1f} tokens")

Docs indexados: 3633 | Vocabulario: 18901 | Longitud promedio: 155.4 tokens


In [51]:
# Ranking y busqueda
def top_k_scores(scores, k):
    return heapq.nlargest(k, scores.items(), key=lambda x: (x[1], x[0]))

In [52]:
#TF-IDF
class TFIDF:
    nombre = "tfidf"

    def __init__(self, indice):
        self.indice = indice
        N = indice.n_docs
        self.idf = {t: math.log(N / len(docs)) for t, docs in indice.postings.items()}
        self.pesos = {
            t: {d: (1.0 + math.log(tf)) * self.idf[t] for d, tf in docs.items()}
            for t, docs in indice.postings.items()
        }
        cuadrados = defaultdict(float)
        for docs in self.pesos.values():
            for d, w in docs.items():
                cuadrados[d] += w * w
        self.normas = {d: math.sqrt(s) for d, s in cuadrados.items()}


modelo_tfidf = TFIDF(indice)
t = next(iter(indice.postings))
d = next(iter(modelo_tfidf.pesos[t]))
print(f"término='{t}' | df={indice.df(t)} | idf={modelo_tfidf.idf[t]:.3f} | "
      f"tf en {d}={indice.postings[t][d]} | w={modelo_tfidf.pesos[t][d]:.3f} | "
      f"||d||={modelo_tfidf.normas[d]:.3f}")


término='statin' | df=27 | idf=4.902 | tf en MED-10=12 | w=17.083 | ||d||=45.023


In [53]:
# Similitud coseo
def vectorizar(modelo, tokens):
    vector = {}
    for t, tf in Counter(tokens).items():
        idf = modelo.idf.get(t, 0.0)
        if idf > 0:
            vector[t] = (1.0 + math.log(tf)) * idf
    return vector

def producto_punto(modelo, vector_q):
    acum = defaultdict(float)
    for t, wq in vector_q.items():
        for d, wd in modelo.pesos[t].items():
            acum[d] += wq * wd
    return acum

def similitud_coseno(modelo, vector_q, acum):
    norma_q = math.sqrt(sum(w * w for w in vector_q.values()))
    if norma_q == 0:
        return {}
    return {d: s / (modelo.normas[d] * norma_q)
            for d, s in acum.items() if modelo.normas.get(d, 0) > 0}

def rankear_tfidf(modelo, tokens, top_k):
    vq = vectorizar(modelo, tokens)
    acum = producto_punto(modelo, vq)
    return top_k_scores(similitud_coseno(modelo, vq, acum), top_k)

TFIDF.rankear = lambda self, tokens, top_k: rankear_tfidf(self, tokens, top_k)

print(modelo_tfidf.rankear(preprocesar("breast cancer diet"), 3))

[('MED-2437', 0.3034112139864074), ('MED-5184', 0.3023488047518953), ('MED-3862', 0.2905433758842112)]


In [54]:
# BM25
class BM25:
    nombre = "bm25"

    def __init__(self, indice, k1=K1, b=B):
        self.indice, self.k1, self.b = indice, k1, b
        N = indice.n_docs
        # IDF de Robertson/Lucene (siempre >= 0)
        self.idf = {t: math.log(1.0 + (N - len(docs) + 0.5) / (len(docs) + 0.5))
                    for t, docs in indice.postings.items()}

    def rankear(self, tokens, top_k):
        idx, k1, b = self.indice, self.k1, self.b
        avgdl = idx.longitud_promedio
        acum = defaultdict(float)
        for t in set(tokens):                     
            if t not in idx.postings:
                continue
            idf = self.idf[t]
            for d, tf in idx.postings[t].items():
                norm_long = 1.0 - b + b * idx.longitudes[d] / avgdl
                acum[d] += idf * tf * (k1 + 1.0) / (tf + k1 * norm_long)
        return top_k_scores(acum, top_k)


modelo_bm25 = BM25(indice)
print(modelo_bm25.rankear(preprocesar("breast cancer diet"), 3))

[('MED-2424', 11.429407226031891), ('MED-2437', 11.317779835655056), ('MED-4450', 11.053049058855681)]


In [55]:
# Busqueda
class Buscador:
    MAX_TOP_K = 100

    def __init__(self, corpus, modelos, preprocesador):
        self.corpus = corpus
        self.modelos = {m.nombre: m for m in modelos}
        self.preprocesar = preprocesador

    def ranking(self, query, model, top_k):
        return self.modelos[model].rankear(self.preprocesar(query), top_k)

    def buscar(self, query, model="bm25", top_k=10, snippet_chars=300):
        if model not in self.modelos:
            raise ValueError(f"model debe ser uno de {sorted(self.modelos)}")
        if not isinstance(query, str) or not query.strip():
            raise ValueError("query no puede estar vacía")
        if not isinstance(top_k, int) or not 1 <= top_k <= self.MAX_TOP_K:
            raise ValueError(f"top_k debe ser un entero entre 1 y {self.MAX_TOP_K}")

        results = []
        for i, (doc_id, score) in enumerate(self.ranking(query, model, top_k), start=1):
            doc = self.corpus[doc_id]
            results.append({
                "rank": i,
                "document_id": doc_id,
                "score": round(float(score), 4),
                "title": doc["titulo"],
                "snippet": doc["texto"][:snippet_chars],
            })
        return {"model": model, "results": results}


buscador = Buscador(corpus, [modelo_tfidf, modelo_bm25], preprocesar)

import json
print(json.dumps(buscador.buscar("breast cancer diet", "tfidf", 2), indent=2, ensure_ascii=False)[:900])

{
  "model": "tfidf",
  "results": [
    {
      "rank": 1,
      "document_id": "MED-2437",
      "score": 0.3034,
      "title": "Diet and breast cancer: understanding risks and benefits.",
      "snippet": "BACKGROUND: Breast cancer is the most commonly diagnosed cancer among women in the United States. Extensive research has been completed to evaluate the relationship between dietary factors and breast cancer risk and survival after breast cancer; however, a summary report with clinical inference is n"
    },
    {
      "rank": 2,
      "document_id": "MED-5184",
      "score": 0.3023,
      "title": "Dietary lignan intakes and risk of breast cancer by tumor estrogen receptor status.",
      "snippet": "We examined the association of dietary lignan intake with estrogen receptor negative (ER-) and ER positive (ER+) breast cancer risk in a breast cancer case-control study. Among preme


In [56]:
# Evaluación
def precision_at_k(ranking, relevantes, k=10):
    return sum(1 for d in ranking[:k] if d in relevantes) / k

def recall_at_k(ranking, relevantes, k=10):
    return sum(1 for d in ranking[:k] if d in relevantes) / len(relevantes)

def ndcg_at_k(ranking, relevantes, k=10):
    # relevantes: {doc_id: grado}; ganancia lineal = grado
    dcg = sum(relevantes.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranking[:k]))
    ideales = sorted(relevantes.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideales))
    return dcg / idcg if idcg > 0 else 0.0

In [57]:
def evaluar(modelo, k=K_EVAL):
    filas = []
    for qid, texto in consultas.items():
        ranking = [d for d, _ in buscador.ranking(texto, modelo, k)]
        rel = qrels[qid]
        filas.append({
            "qid": qid,
            "P@10": precision_at_k(ranking, rel, k),
            "R@10": recall_at_k(ranking, rel, k),
            "NDCG@10": ndcg_at_k(ranking, rel, k),
        })
    return pd.DataFrame(filas).set_index("qid")

def validar(resp, top_k):
    r = resp["results"]
    ids = [x["document_id"] for x in r]
    scores = [x["score"] for x in r]
    assert len(ids) <= top_k, "excede top_k"
    assert len(ids) == len(set(ids)), "hay duplicados"
    assert scores == sorted(scores, reverse=True), "no está ordenado"
    assert all(i in corpus for i in ids), "ID inexistente en el corpus"
    assert [x["rank"] for x in r] == list(range(1, len(ids) + 1)), "ranks incorrectos"

In [58]:
libres = ["vitamin d and bone health", "effect of green tea on blood pressure", "xyzzy qwerty"]
pruebas = list(consultas.values())[:20] + libres

for modelo in ["tfidf", "bm25"]:
    for k in [1, 5, 10, 50]:
        for q in pruebas:
            validar(buscador.buscar(q, modelo, k), k)
print("Validaciones de formato: OK")

for args in [("", "bm25", 10), ("cancer", "dense", 10), ("cancer", "bm25", 0), ("cancer", "bm25", 1000)]:
    try:
        buscador.buscar(*args)
        print("NO falló:", args)
    except ValueError as e:
        print("OK ->", e)

for m in ["tfidf", "bm25"]:
    r = buscador.buscar("effect of green tea on blood pressure", m, 3)
    print(m, [(x["document_id"], x["score"]) for x in r["results"]])

Validaciones de formato: OK
OK -> query no puede estar vacía
OK -> model debe ser uno de ['bm25', 'tfidf']
OK -> top_k debe ser un entero entre 1 y 100
OK -> top_k debe ser un entero entre 1 y 100
tfidf [('MED-5047', 0.4284), ('MED-4097', 0.3468), ('MED-4365', 0.3207)]
bm25 [('MED-5047', 19.2065), ('MED-1842', 16.8315), ('MED-4365', 16.3858)]


## Consultas usadas
Se utilizó las siguientes consultas propias para validar el formato de respuestas y hacer una evaluación rápida de los modelos:
- vitamin d and bone health
- effect of green tea on blood presure
- xyzzy qwerty

Y tambien se utilizaron las 20 primeras consultas del los qrels para las evaluaciones.

In [59]:
print(f"Partición: '{SPLIT_QRELS}' | Consultas evaluadas: {len(consultas)}")
print("Primeras 10 consultas:")
for qid in list(consultas)[:10]:
    print(f"  {qid}: {consultas[qid]}  ({len(qrels[qid])} relevantes)")

res_tfidf = evaluar("tfidf")
res_bm25 = evaluar("bm25")

resumen = pd.DataFrame({
    "TF-IDF (coseno)": res_tfidf.mean(),
    "BM25": res_bm25.mean(),
}).T.round(4)
display(resumen)

Partición: 'test' | Consultas evaluadas: 323
Primeras 10 consultas:
  PLAIN-2: Do Cholesterol Statin Drugs Cause Breast Cancer?  (24 relevantes)
  PLAIN-12: Exploiting Autophagy to Live Longer  (30 relevantes)
  PLAIN-23: How to Reduce Exposure to Alkylphenols Through Your Diet  (90 relevantes)
  PLAIN-33: What’s Driving America’s Obesity Problem?  (32 relevantes)
  PLAIN-44: Who Should be Careful About Curcumin?  (58 relevantes)
  PLAIN-56: Foods for Glaucoma  (50 relevantes)
  PLAIN-68: What is Actually in Chicken Nuggets?  (55 relevantes)
  PLAIN-78: What Do Meat Purge and Cola Have in Common?  (62 relevantes)
  PLAIN-91: Chronic Headaches and Pork Parasites  (64 relevantes)
  PLAIN-102: Stopping Heart Disease in Childhood  (24 relevantes)


,P@10,R@10,NDCG@10
TF-IDF (coseno),0.2297,0.1489,0.3173
BM25,0.2393,0.1528,0.3272


In [60]:
dif = res_bm25["NDCG@10"] - res_tfidf["NDCG@10"]
print(f"BM25 gana:   {(dif > 0).sum()} consultas")
print(f"TF-IDF gana: {(dif < 0).sum()} consultas")
print(f"Empates:     {(dif == 0).sum()} consultas")

for m in ["P@10", "R@10", "NDCG@10"]:
    try:
        _, p = wilcoxon(res_bm25[m], res_tfidf[m])
        print(f"{m}: p-valor Wilcoxon = {p:.4g}")
    except ValueError:
        print(f"{m}: diferencias nulas, no aplica Wilcoxon")

comp = pd.DataFrame({
    "query": pd.Series(consultas),
    "n_rel": pd.Series({q: len(d) for q, d in qrels.items()}),
    "NDCG_tfidf": res_tfidf["NDCG@10"],
    "NDCG_bm25": res_bm25["NDCG@10"],
})
comp["dif"] = comp["NDCG_bm25"] - comp["NDCG_tfidf"]
pd.set_option("display.max_colwidth", 60)
print("\nMayor ventaja de TF-IDF:")
display(comp.sort_values("dif").head(5).round(3))
print("Mayor ventaja de BM25:")
display(comp.sort_values("dif").tail(5).round(3))

print("Consultas con R@10 = 0 -> TF-IDF:", (res_tfidf["R@10"] == 0).sum(),
      "| BM25:", (res_bm25["R@10"] == 0).sum())

BM25 gana:   90 consultas
TF-IDF gana: 60 consultas
Empates:     173 consultas
P@10: p-valor Wilcoxon = 0.002771
R@10: p-valor Wilcoxon = 0.003913
NDCG@10: p-valor Wilcoxon = 0.002724

Mayor ventaja de TF-IDF:


,query,n_rel,NDCG_tfidf,NDCG_bm25,dif
PLAIN-3332,Alcohol Risks vs. Benefits,14,0.729,0.429,-0.300
PLAIN-956,cooking methods,206,0.676,0.458,-0.217
PLAIN-1601,memory,74,0.782,0.581,-0.201
PLAIN-3151,Too Much Iodine Can Be as Bad as Too Little,18,0.731,0.545,-0.185
PLAIN-3292,Are Multivitamins Good For You?,7,0.661,0.489,-0.172


Mayor ventaja de BM25:


,query,n_rel,NDCG_tfidf,NDCG_bm25,dif
PLAIN-701,black raspberries,8,0.379,0.584,0.204
PLAIN-2680,Counteracting the Effects of Dioxins Through Diet,14,0.091,0.328,0.238
PLAIN-2660,How Long to Detox From Fish Before Pregnancy?,27,0.110,0.362,0.252
PLAIN-2770,Flaxseeds & Breast Cancer Survival: Clinical Evidence,41,0.225,0.485,0.260
PLAIN-1667,muscle health,159,0.470,0.767,0.297


Consultas con R@10 = 0 -> TF-IDF: 104 | BM25: 98


## Interpretación
- BM25 supera a TF-IDF en las tres métricas: P@10 0.2393 vs 0.2297, R@10 0.1528 vs 0.1489 y NDCG@10 0.3272 vs 0.3173. Por consulta, BM25 gana en 90, TF-IDF en 60 y hay 173 empates. Podemos decir que a pesar de que las métricas sean muy similares, BM25 es sustancialmente mas eficaz.
- BM25 tiene mejores resltados porque satura la frecuencia del término y normaliza explícitamente por longitud. En NFCorpus la longitud de los documentos varía mucho.
- Muchas consultas tienen más de 10 relevantes, lo que limita P@10 y R@10.
- Se podría obtener mejores resultados si se hiciera un mejor ajuste del ki y del b para BM25. Sin embargo tambien se podría realizar expanciones de consultas ya que el corpus probablemente tiene muchos sinónimos para los mismos fenomenos médicos que redactan los documentos.


In [ ]:
# Servicio web exportar para servicio web
import gzip, json, pathlib

SALIDA = pathlib.Path("servicio/datos")
SALIDA.mkdir(parents=True, exist_ok=True)

artefacto = {
    "stopwords": sorted(STOPWORDS),
    "longitudes": indice.longitudes,                       # ID
    "postings": indice.postings,                           # termino 
    "docs": {d: [doc["titulo"], doc["texto"][:300]]        # totulo + snippet 
             for d, doc in corpus.items()},
}
ruta = SALIDA / "indice.json.gz"
with gzip.open(ruta, "wt", encoding="utf-8") as f:
    json.dump(artefacto, f, ensure_ascii=False, separators=(",", ":"))
print(f"Índice exportado: {ruta} ({ruta.stat().st_size/1e6:.1f} MB) | "
      f"{len(artefacto['docs'])} docs | {len(artefacto['postings'])} términos")

Índice exportado: servicio\datos\indice.json.gz (1.5 MB) | 3633 docs | 18901 términos
